# Problem Set 2, Exercise 3
Pryce Davies and Richard Chen.  

 Assuming that $\epsilon_t\stackrel{i.i.d}{\sim}N(0,1)$, use the data in ps2\_ex3.csv to estimate the parameters $(\beta,\phi,\delta)$ using MLE.  
 \
 Likelihood Function (for $j=2$):
$$
\mathcal{L}(n_t)
= \prod_{t=1}^T \Bigg(
\mathbf{1}\{n_t = 0\}\, \mathbb{P}(\epsilon_t < \phi - x_t \beta)

+ \mathbf{1}\{n_t = 1\}\,
\mathbb{P}\!\left(\phi - x_t \beta \le \epsilon_t < \phi + \delta \ln(2) - x_t \beta\right)
\\
+ \mathbf{1}\{n_t = 2\}\,
\mathbb{P}\!\left(\epsilon_t \ge \phi + \delta \ln(2) - x_t \beta\right)
\Bigg)
$$

In [98]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize

In [99]:
# Read in Data
df = pd.read_csv('data/ps2_ex3.csv')
max(df['n'])

24

In [100]:
# Estimate probability given \beta, \phi, \delta
def cond_prob_theta(data,theta):
    beta = theta[0]
    phi = theta[1]
    delta = theta[2]
    # initialize list of probs
    prob = pd.DataFrame(np.zeros((len(data['x']),max(data['n'])+1)))
    # compute bounds and probabiliites for each choice
    for i in range(max(data['n'])+1):
        if i == 0:
            prob.iloc[:,i]  = norm.cdf(phi- beta * data['x'])    
        elif i == max(data['n']):
            prob.iloc[:,i]  = 1-norm.cdf(phi- beta * data['x']+ delta * np.log(24))    
        else:
            prob.iloc[:,i]  = norm.cdf(phi- beta * data['x']+ delta *np.log(i+1)) - norm.cdf(phi- beta * data['x']+delta *np.log(i)) 

    return prob


In [101]:
# Likelihood function (adapted from notebook)
def likelihood(data, prob):
    """The likelihood function with the given choice probability
    
        Parameters
        ----------
        data : `DataFrame`
            The data that contains the actions and states.
        dgp : `dgp`
            The primitives of the DDC model.
        prob : `ndarray`
            A matrix of choice probabilities.
    """
    
    l = 0.
    for i in range(len(data['x'])):
        num_enter = data['n'].iloc[i]     
        # Append the right probability depending on the choice and state
        l += np.log(prob.iloc[i, num_enter]) 
            
    return -l/(len(data['x'])*max(data['n']))

In [102]:
# MLE Estimation
def entry_mle(data):
    """
    The two-step CCP approach.

        Parameters
        ----------
        data : `DataFrame`
            The data that contains the actions and states.
        dgp : `dgp`
            The primitives of the DDC model.
    """
    def ll(x):
        prob = cond_prob_theta(data,x)
        return likelihood(data, prob)
    θ = minimize(ll, [1, 1, 1])
    return θ

In [104]:
out = entry_mle(df)
out.x

array([ 2.14772129,  1.65083203, 10.50923141])

In [111]:
table = cond_prob_theta(df,out.x)
df['MLE_implied_num'] = table.idxmax(axis=1)
np.mean(df['MLE_implied_num'] == df['n'])

np.float64(0.75)